# morning-gemma v2: SFT + 선호 학습(DPO) → v1과 비교 → 폰용 변환 (Kaggle GPU 한 세션)

**설정**: Accelerator = **GPU T4**, Internet **on**. Input 두 개:
- `morning-gemma-merged` (v1 합친 모델, 비교용)
- `morning-call-v2-data` (`engine/`, `tools/finetune/`의 학습 데이터와 `eval_stream.py`)

| 단계 | 하는 일 |
|---|---|
| 1 | 설치, 데이터 읽기 |
| 2 | 원본 모델(4비트) + LoRA로 SFT (v1과 같은 설정, 데이터만 늘림) → **v2-SFT 시험** |
| 3 | 선호 학습(DPO): 검사를 통과한 대답(chosen)이 걸린 대답(rejected)보다 나오기 쉽게 → **v2 시험** |
| 4 | 합쳐서 저장(`merged_v2`) → v1을 불러와 **같은 시험** → 비교 표 |
| 5 | 폰용 변환 (`kaggle_convert_ple.ipynb`와 같은 방법). 메모리가 모자라면 **커널을 다시 시작하고 5번부터** 실행 |

v2가 v1보다 나을 때만 Hugging Face의 `manifest.json` version을 올린다 (앱에 "새 모델 받기"가 뜬다).


In [ ]:
import os, glob, sys, json, random, subprocess, gc, time
os.environ["UNSLOTH_RETURN_LOGITS"] = "0"   # 1이면 SFT 중간 평가가 어휘 26만 개 logits를 다 만들어 GPU 메모리가 모자란다 (DPO 칸에서만 1)
CODE = os.path.dirname(glob.glob("/kaggle/input/**/tools/finetune/eval_stream.py", recursive=True)[0])
ROOT = os.path.dirname(os.path.dirname(CODE))
V1 = os.path.dirname([p for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if "morning-gemma-merged" in p][0])
sys.path.insert(0, f"{ROOT}/engine"); sys.path.insert(0, CODE)
print("code:", ROOT, "\nv1:", V1)
!nvidia-smi --query-gpu=name,memory.total --format=csv; free -g | head -2
!pip install -q unsloth 2>&1 | tail -2


In [ ]:
import dialog, eval_stream
def load(name):
    p = f"{CODE}/{name}"
    return [json.loads(l) for l in open(p, encoding="utf-8")] if os.path.exists(p) else []
# wake_train: v1 데이터 · more: DPO 데이터를 만들며 이어 간 통화 · public: 공개 데이터 문장을 딴 얘기로 넣은 통화
calls = load("wake_train.jsonl") + load("wake_train_more.jsonl") + load("wake_train_public.jsonl")
random.Random(0).shuffle(calls)
n_val = max(10, len(calls) // 10)
val_calls, train_calls = calls[:n_val], calls[n_val:]

# DPO 짝: '깨움 표현 없음'만 걸린 짝은 정규식 차이(예: '눈 좀 떠' vs '눈을 살짝 떠')인 경우가 많아, 다른 문제가 있는 짝과 1:1까지만 쓴다
pairs = [p for p in load("wake_dpo.jsonl") + load("wake_dpo_public.jsonl") if p["chosen"] != p["rejected"]]
presence_only = [p for p in pairs if p["issues"] == ["깨우는 중이라는 표현 없음"]]
others = [p for p in pairs if p["issues"] != ["깨우는 중이라는 표현 없음"]]
random.Random(1).shuffle(presence_only)
dpo = others + presence_only[:len(others)]
random.Random(2).shuffle(dpo)
dpo = dpo[:1000]
from collections import Counter
print(f"SFT 통화 {len(calls)} (학습 {len(train_calls)}, 확인 {len(val_calls)}) · DPO 짝 {len(dpo)} / 전체 {len(pairs)}")
print(Counter(i for p in dpo for i in p["issues"]).most_common())


## 2. 원본 모델 + SFT (v1과 같은 설정)

In [ ]:
from unsloth import FastModel
import torch, torch.nn.functional as F
MAX_LEN = 4096
model, processor = FastModel.from_pretrained("unsloth/gemma-4-E2B-it", max_seq_length=MAX_LEN, load_in_4bit=True, full_finetuning=False)
tok = getattr(processor, "tokenizer", processor)

def encode(messages):
    text = tok.apply_chat_template(messages, tokenize=False, enable_thinking=False)
    spans = []
    for i, m in enumerate(messages):
        if m["role"] != "assistant":
            continue
        before = tok.apply_chat_template(messages[:i], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        upto = tok.apply_chat_template(messages[:i + 1], tokenize=False, enable_thinking=False)
        assert text.startswith(before) and text.startswith(upto)
        spans.append((len(before), len(upto)))
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True, truncation=True, max_length=MAX_LEN)
    labels = [t if any(a <= s < b for a, b in spans) else -100 for t, (s, e) in zip(enc["input_ids"], enc["offset_mapping"])]
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}

from datasets import Dataset
train_ds = Dataset.from_list([encode(c["messages"]) for c in train_calls])
val_ds = Dataset.from_list([encode(c["messages"]) for c in val_calls])
model = FastModel.get_peft_model(model, finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True, r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407)
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bf16_supported
FastModel.for_training(model)
trainer = Trainer(model=model, train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
    args=TrainingArguments(output_dir="lora-out", per_device_train_batch_size=1, gradient_accumulation_steps=8,
        num_train_epochs=2, learning_rate=1e-4, warmup_steps=10, lr_scheduler_type="linear",
        logging_steps=5, eval_strategy="steps", eval_steps=25, save_strategy="no",
        bf16=is_bf16_supported(), fp16=not is_bf16_supported(), optim="adamw_8bit",
        gradient_checkpointing=True, report_to="none", seed=3407))
t0 = time.time(); trainer.train(); print(f"SFT {time.time()-t0:.0f}초")
model.save_pretrained("lora_v2_sft")


In [ ]:
# 앱(PhoneAi.kt)과 같은 생성 설정: 생각 끔, temperature 0.7, top_k 40, top_p 0.95, 최대 120토큰
def hf_stream(msgs):
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=120, do_sample=True, temperature=0.7, top_p=0.95, top_k=40)
    yield tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

RESULTS = {}
FastModel.for_inference(model)
t0 = time.time(); RESULTS["v2-SFT"] = eval_stream.evaluate(hf_stream, repeats=2)
print(RESULTS["v2-SFT"][0], f"{time.time()-t0:.0f}초")


## 3. 선호 학습 (DPO)
같은 상황에서 **검사를 통과한 대답(chosen)**과 **걸린 대답(rejected)**을 보여 주고, chosen 쪽 확률을 기준 모델(SFT 직후)보다 더 올리게 한다.
`loss = -log σ(β·[(log p(chosen) - log p_ref(chosen)) - (log p(rejected) - log p_ref(rejected))])`, β=0.1
- 기준 모델 확률은 학습 전에 한 번 계산해 둔다 (같은 LoRA를 이어서 학습하므로 모델을 하나 더 둘 필요가 없다).
- `정확도`: chosen이 rejected보다 더 올라간 짝의 비율. 50%보다 커지면 배우는 중이다.


In [ ]:
def prompt_completion(prompt, completion):
    p = tok.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    full = tok.apply_chat_template(prompt + [{"role": "assistant", "content": completion}], tokenize=False, enable_thinking=False)
    assert full.startswith(p)
    pi = tok(p, add_special_tokens=False)["input_ids"]
    ci = tok(full[len(p):], add_special_tokens=False)["input_ids"]
    return pi + ci, len(pi)

def seq_logp(ids, n_prompt):
    x = torch.tensor([ids], device=model.device)
    k = len(ids) - n_prompt + 1
    try:
        out = model(input_ids=x, logits_to_keep=k)   # 대답 부분 logits만 (어휘 26만 개라 전체는 메모리를 많이 쓴다)
    except TypeError:
        out = model(input_ids=x)
    logits = out.logits[0, -k:-1].float()            # 대답 토큰들을 맞히는 위치
    tgt = x[0, n_prompt:]
    return torch.log_softmax(logits, -1).gather(-1, tgt[:, None]).sum()

os.environ["UNSLOTH_RETURN_LOGITS"] = "1"   # 확률을 직접 계산하려면 logits가 필요하다 (logits_to_keep으로 대답 부분만)
data = [(prompt_completion(p["prompt"], p["chosen"]), prompt_completion(p["prompt"], p["rejected"])) for p in dpo]
data = [d for d in data if max(len(d[0][0]), len(d[1][0])) <= MAX_LEN]
FastModel.for_training(model); model.eval()
ref = []
t0 = time.time()
with torch.no_grad():
    for i, (c, r) in enumerate(data):
        ref.append((seq_logp(*c).item(), seq_logp(*r).item()))
        if i % 100 == 0: print(f"기준 확률 {i}/{len(data)} {time.time()-t0:.0f}초", flush=True)
print("기준 확률 끝", f"{time.time()-t0:.0f}초")


In [ ]:
BETA, LR, ACCUM = 0.1, 2e-5, 8
FastModel.for_training(model); model.train()
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR)
order = list(range(len(data))); random.Random(3).shuffle(order)
acc, losses, t0 = [], [], time.time()
for step, i in enumerate(order):
    (c, r), (rc, rr) = data[i], ref[i]
    margin = (seq_logp(*c) - rc) - (seq_logp(*r) - rr)
    loss = -F.logsigmoid(BETA * margin)
    (loss / ACCUM).backward()
    acc.append(float(margin.item() > 0)); losses.append(loss.item())
    if (step + 1) % ACCUM == 0 or step == len(order) - 1:
        torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); opt.zero_grad()
    if (step + 1) % 80 == 0:
        print(f"{step+1}/{len(order)} loss {sum(losses[-80:])/80:.3f} 정확도 {sum(acc[-80:])/80:.0%} {time.time()-t0:.0f}초", flush=True)
model.save_pretrained("lora_v2")
FastModel.for_inference(model)
t0 = time.time(); RESULTS["v2"] = eval_stream.evaluate(hf_stream, repeats=2)
print(RESULTS["v2"][0], f"{time.time()-t0:.0f}초")


## 4. 합쳐서 저장 → v1과 같은 시험 → 비교 표

In [ ]:
model.save_pretrained_merged("merged_v2", processor, save_method="merged_16bit")
!du -sh merged_v2
del model, trainer, opt; gc.collect(); torch.cuda.empty_cache()
model, processor = FastModel.from_pretrained(V1, max_seq_length=MAX_LEN, load_in_4bit=True)
tok = getattr(processor, "tokenizer", processor)
FastModel.for_inference(model)
t0 = time.time(); RESULTS["v1"] = eval_stream.evaluate(hf_stream, repeats=2)
print(RESULTS["v1"][0], f"{time.time()-t0:.0f}초")
report = eval_stream.markdown({k: RESULTS[k] for k in ["v1", "v2-SFT", "v2"]})
open("eval_v1_v2.md", "w", encoding="utf-8").write("# v1 vs v2 (Kaggle T4, 4비트로 불러와 같은 대화 엔진으로 시험)\n\n" + report)
print(report.split("\n## ")[0])
del model; gc.collect(); torch.cuda.empty_cache()


## 5. 폰용 변환 (`kaggle_convert_ple.ipynb`와 같은 방법)
공식 `per_layer_embedder` 재사용 + PLE 표 메모리에서 지움 + 가벼운 변환 + PLE ×16 보정. 약 25분.
메모리 부족으로 커널이 다시 시작되면 **이 칸부터** 다시 실행하면 된다 (`merged_v2`는 남아 있다).

In [ ]:
import os, glob, subprocess, json
MERGED = "/kaggle/working/merged_v2"
# 출력 폴더(/kaggle/working)는 19.5GB까지다. merged_v2(9.6GB) 옆에 학습 중간 파일이 쌓여 변환 결과를 쓸 자리가 없었다
!du -sh /kaggle/working/* /kaggle/working/.[!.]* 2>/dev/null | sort -h | tail -8
!rm -rf /kaggle/working/lora-out /kaggle/working/.cache /kaggle/working/unsloth_compiled_cache /kaggle/working/out
!du -sh /kaggle/working; df -h /kaggle/working /tmp | tail -2
!pip install -q uv
!uv tool install -q litert-torch-nightly
!uv tool install -q litert-lm
os.environ["PATH"] += ":/root/.local/bin"
PY = subprocess.check_output(["uv", "tool", "dir"], text=True).strip() + "/litert-torch-nightly/bin/python"
print("PY:", PY, os.path.exists(PY)); print(sorted(os.listdir(MERGED)))


In [ ]:
%%writefile /tmp/inspect.py
# .litertlm을 풀어서 구성 파일과 각 tflite의 신호(signature) 이름을 보여 준다.
import sys, os, json
from litert_torch.generative.export_hf.experimental.litertlm_bundle import litertlm_bundle as L
src, dst = sys.argv[1], sys.argv[2]
comp = L.unpack_litertlm(src, dst)
sigs = {}
for k, v in comp.items():
    size = os.path.getsize(v) if isinstance(v, str) and os.path.isfile(v) else ""
    print(f"{k:32s} {size:>12} {v}")
    if isinstance(v, str) and v.endswith(".tflite"):
        try:
            from ai_edge_litert.interpreter import Interpreter
            sl = Interpreter(model_path=v).get_signature_list()
            sigs[k] = {n: sorted(d["inputs"]) for n, d in sl.items()}
        except Exception as e:
            sigs[k] = f"(signature 읽기 실패: {e})"
json.dump({"components": {k: str(v) for k, v in comp.items()}, "sigs": sigs}, open(dst + "/summary.json", "w"), indent=1, ensure_ascii=False)
for k, s in sigs.items():
    print("==", k); print(json.dumps(s, ensure_ascii=False)[:1500])


In [ ]:
!{PY} -c "from huggingface_hub import hf_hub_download as d; print(d('litert-community/gemma-4-E2B-it-litert-lm','gemma-4-E2B-it.litertlm',local_dir='/tmp/official'))"
if not os.path.exists("/tmp/official_unpacked/summary.json"):
    !rm -rf /tmp/official_unpacked && {PY} /tmp/inspect.py /tmp/official/gemma-4-E2B-it.litertlm /tmp/official_unpacked 2>&1 | grep -v Warning | tail -40
import json
OFFICIAL_PLE = json.load(open("/tmp/official_unpacked/summary.json"))["components"]["tf_lite_per_layer_embedder"]
print("OFFICIAL_PLE:", OFFICIAL_PLE)

In [ ]:
%%writefile /tmp/export_reuse_ple.py
# per_layer_embedder 변환 단계만 "공식 파일 경로 끼워 넣기"로 바꾸고 나머지는 원래 도구 그대로 돌린다.
import sys, dataclasses
merged, official_ple = sys.argv[1], sys.argv[2]
from litert_torch.generative.export_hf.core import export_lib
def reuse_official_ple(source_model_artifacts, export_config, exported_model_artifacts):
    print(">>> per_layer_embedder: 변환 건너뜀, 공식 파일 사용:", official_ple, flush=True)
    # 본체(prefill/decode)는 이 단어 표를 입력(per_layer_embeddings)으로 받으므로 표 자체는 필요 없다.
    # 32비트로 약 9.4GB라서 본체 변환 전에 메모리에서 지운다.
    import gc, torch
    for name, mod in source_model_artifacts.model.named_modules():
        if name.endswith("embed_tokens_per_layer") and hasattr(mod, "weight"):
            n = mod.weight.numel()
            mod.weight = torch.nn.Parameter(torch.empty(0, mod.weight.shape[1]), requires_grad=False)
            print(f">>> 메모리에서 지움: {name} ({n * 4 / 1e9:.1f}GB)", flush=True)
    gc.collect()
    paths = dict(exported_model_artifacts.additional_model_paths or {})
    paths["per_layer_embedder"] = official_ple
    return dataclasses.replace(exported_model_artifacts, additional_model_paths=paths)
export_lib.export_additional_models = reuse_official_ple
# 11/21 3차: 공식 PLE 출력은 "표 값 그대로"(배율 1)이고, 우리 본체는 "표 x sqrt(256)=16"을 기대한다
# (점검 칸: 공식 출력 std 0.062 vs 표x16 std 1.07, 층별 상관 0.8~0.94). 그대로 두면 대답이 이모지로 깨진다.
# → 본체(prefill/decode)가 받은 per_layer_embeddings에 sqrt(d)를 곱하게 한다.
import inspect, json as _json, math
from litert_torch.generative.export_hf.model_ext.gemma4 import exportable_module as g4
_cfg = _json.load(open(merged + "/config.json")); _tc = _cfg.get("text_config", _cfg)
PLE_SCALE = math.sqrt(_tc["hidden_size_per_layer_input"])
def _scaled(orig):
    def adapt_inputs(self, tokens, embeddings, per_layer_embeddings, *a, **k):
        return orig(self, tokens, embeddings, per_layer_embeddings * PLE_SCALE, *a, **k)
    return adapt_inputs
for _n, _c in list(vars(g4).items()):
    if inspect.isclass(_c) and "adapt_inputs" in _c.__dict__ and "per_layer_embeddings" in inspect.signature(_c.__dict__["adapt_inputs"]).parameters:
        _c.adapt_inputs = _scaled(_c.__dict__["adapt_inputs"]); print(f">>> PLE x{PLE_SCALE} 패치: {_n}", flush=True)
from litert_torch.generative.export_hf import export as lib
# 앱은 글자만 쓰므로 그림(vision)·소리(audio) 인코더는 만들지 않는다.
# (use_v2=True는 메모리가 4GB로 줄었지만 tf-nightly와 버전이 안 맞아 MLIR 오류가 났다 → 기존 변환기 사용)
lib.export(model=merged, output_dir="/tmp/out", externalize_embedder=True,
           jinja_chat_template_override="litert-community/gemma-4-E2B-it-litert-lm",
           export_vision_encoder=False, export_audio_encoder=False,
           # 11/21: 31GB에서 마지막 "Merge MLIR Modules"(prefill_128+prefill_1024+decode 합치기) 중 메모리 부족.
           # → prefill_1024를 빼서 합칠 덩어리를 줄인다. 긴 입력은 런타임이 128씩 나눠 넣는다(NPU 설정도 128 하나만 씀).
           # → 캐시 길이도 32771 → 4096. 앱 런타임은 원래 4096 토큰까지만 쓰므로(max_num_tokens 4096) 손해 없음.
           prefill_lengths=[128], cache_length=4096,
           # 11/21 2차: prefill_1024를 빼도 Merge 직후 29GB로 실패 → 가중치를 MLIR에 복사하지 않는 방식.
           # (11/20엔 이 옵션이 PLE 파일 쓰기에서 실패했지만, 지금은 PLE를 공식 파일로 건너뛴다)
           experimental_lightweight_conversion=True)


In [ ]:
# 메모리 사용량을 1분마다 /tmp/mem.log에 남긴다 (어디서 메모리가 모자라는지 보려고)
subprocess.Popen("while true; do date +%T; free -g | sed -n 2p; sleep 60; done > /tmp/mem.log 2>&1", shell=True)
!rm -rf /tmp/out && {PY} /tmp/export_reuse_ple.py {MERGED} {OFFICIAL_PLE} 2>&1 | grep --line-buffered -E "FAIL|Error|error|>>>|Traceback|\(\+[0-9]"
!ls -la /tmp/out; tail -5 /tmp/mem.log

In [ ]:
!timeout 900 litert-lm run /tmp/out/model.litertlm --prompt="5분만 더 잘래" 2>&1 | tail -3 | cut -c1-400
!timeout 900 litert-lm run /tmp/out/model.litertlm --prompt="목이 너무 말라" 2>&1 | tail -3 | cut -c1-400
!sha256sum /tmp/out/model.litertlm; stat -c %s /tmp/out/model.litertlm
!cp /tmp/out/model.litertlm /kaggle/working/morning-gemma-v2.litertlm && ls -la /kaggle/working/*.litertlm
